In [0]:
# Databricks notebook source
# COMMAND ----------
# 1) Config

from __future__ import annotations

import json
import re
import time
from typing import Any, Dict, List, Optional

import pandas as pd
from pyspark.sql import Window
from pyspark.sql import functions as F

SOURCE_TABLE = "kic_data_ods.buzzmetrix.buzzmetrix"
ENDPOINT = "databricks-gpt-5-4"

PROMPT_VERSION = "category_topic_dynamic_rules_allgroups_v1"
SAVE_DB = "sandbox.z_jungryo_lee"

RULE_PROFILE_TABLE = f"{SAVE_DB}.category_topic_rule_profile_{PROMPT_VERSION}"
TOPIC_POOL_TABLE = f"{SAVE_DB}.category_topic_pool_{PROMPT_VERSION}"
DETAIL_TABLE = f"{SAVE_DB}.category_topic_detail_{PROMPT_VERSION}"
SUMMARY_TABLE = f"{SAVE_DB}.category_topic_summary_{PROMPT_VERSION}"

MIN_GROUP_SIZE = 100
MAX_SAMPLE_ROWS = 1000
MAX_RULE_SAMPLE_ROWS = 800
MAX_FINAL_TOPICS = 17
MIN_FINAL_TOPICS = 7
MIN_ROWS_PER_TOPIC = 10
CLASSIFY_BATCH_SIZE = 25

MAX_TOKENS = 2200
MAX_RETRIES = 3
BACKOFF_BASE = 1.8
RATE_LIMIT_SECONDS = 0.4

# None 이면 전체 그룹, 숫자 넣으면 일부 그룹만 테스트 가능
LIMIT_GROUP_COUNT = None

print(f"[CONFIG] endpoint={ENDPOINT}, prompt_version={PROMPT_VERSION}, min_group_size={MIN_GROUP_SIZE}")
print(f"[CONFIG] sample_rows={MAX_SAMPLE_ROWS}, rule_sample_rows={MAX_RULE_SAMPLE_ROWS}, limit_group_count={LIMIT_GROUP_COUNT}")


In [0]:
# COMMAND ----------
# 2) Helpers

def clean_text(x: Any) -> str:
    return "" if x is None else re.sub(r"\s+", " ", str(x)).strip()


def extract_json(text: str) -> Dict[str, Any]:
    text = clean_text(text)

    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        candidate = match.group(0)
        try:
            return json.loads(candidate)
        except Exception:
            candidate = re.sub(r",\s*}", "}", candidate)
            candidate = re.sub(r",\s*]", "]", candidate)
            return json.loads(candidate)

    raise ValueError(f"Invalid JSON: {text[:1000]}")


def chunk_list(items: List[Any], size: int) -> List[List[Any]]:
    return [items[i:i + size] for i in range(0, len(items), size)]


def sc_label(sc_measurement: int) -> str:
    if sc_measurement == 1:
        return "긍정"
    if sc_measurement == -1:
        return "부정"
    return "기타"


def overall_label(sc_measurement: int) -> str:
    if sc_measurement == 1:
        return "전반적 긍정"
    if sc_measurement == -1:
        return "전반적 부정"
    return "전반적 평가"


def call_llm(messages: List[Dict[str, str]], max_tokens: int = MAX_TOKENS) -> Dict[str, Any]:
    from mlflow.deployments import get_deploy_client

    client = get_deploy_client("databricks")
    payload = {
        "messages": messages,
        "temperature": 0.0,
        "max_tokens": max_tokens,
    }

    last_error: Optional[Exception] = None
    for attempt in range(MAX_RETRIES):
        resp = None
        try:
            print(f"[LLM CALL] attempt={attempt + 1}/{MAX_RETRIES}")
            resp = client.predict(endpoint=ENDPOINT, inputs=payload)

            if isinstance(resp, dict):
                if "choices" in resp and resp["choices"]:
                    return extract_json(resp["choices"][0]["message"]["content"])
                if "predictions" in resp and resp["predictions"]:
                    pred0 = resp["predictions"][0]
                    if isinstance(pred0, dict) and "content" in pred0:
                        return extract_json(pred0["content"])
                    if isinstance(pred0, str):
                        return extract_json(pred0)
                if "content" in resp:
                    return extract_json(resp["content"])

            if isinstance(resp, str):
                return extract_json(resp)

            raise ValueError(f"Unexpected response schema: {resp}")

        except Exception as exc:
            last_error = exc
            print(f"[LLM ERROR] attempt={attempt + 1}/{MAX_RETRIES}, error={repr(exc)}")
            if resp is not None:
                print("[LLM RAW RESPONSE PREVIEW]")
                print(str(resp)[:2000])
            wait_sec = BACKOFF_BASE ** attempt
            print(f"[LLM RETRY WAIT] {wait_sec:.2f}s")
            time.sleep(wait_sec)

    raise RuntimeError(f"LLM call failed: {repr(last_error)}")


def save_table(df, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        print(f"[SAVE] overwrite -> {table_name}")
    else:
        print(f"[SAVE] create -> {table_name}")
    df.write.mode("overwrite").format("delta").saveAsTable(table_name)


In [0]:
# COMMAND ----------
# 3) Category Feature Patterns

COMMON_FEATURE_PATTERNS = [
    "quality", "picture", "image", "visual", "sound", "audio", "video", "screen", "display", "tv",
    "setup", "install", "installation", "manual", "guide", "weight", "stand", "mount", "wall",
    "cable", "port", "hdmi", "bluetooth", "wifi", "wireless", "connect", "connection", "compatibility",
    "os", "software", "menu", "ui", "app", "apps", "channel", "content", "voice", "game", "gaming",
    "iot", "mobile", "brand", "service", "support", "warranty", "energy", "efficiency", "price",
    "value", "design", "color", "material", "finish", "heat", "durability", "panel", "glare",
    "angle", "brightness", "contrast", "resolution", "sharp", "clarity", "bass", "surround",
    "화질", "음질", "사운드", "화면", "디스플레이", "설치", "세팅", "매뉴얼", "가이드", "무게",
    "스탠드", "벽걸이", "선정리", "단자", "블루투스", "와이파이", "연결", "호환", "소프트웨어",
    "메뉴", "ui", "앱", "채널", "콘텐츠", "음성", "게임", "iot", "모바일", "브랜드", "서비스",
    "보증", "에너지", "효율", "가격", "가성비", "디자인", "색상", "소재", "마감", "발열",
    "내구성", "패널", "반사", "시야각", "밝기", "명암", "해상도", "선명", "저음", "서라운드"
]

CATEGORY_FEATURE_PATTERNS = {
    ("01. 사이즈", "01-01. TV 사이즈"): [
        "size", "inch", "inches", "big", "large", "small", "fit", "fits",
        "사이즈", "크기", "인치", "대형", "소형", "공간", "잘맞"
    ],
    ("02. 화질", "02-01. 선명도"): [
        "sharp", "sharpness", "clear", "clarity", "crisp", "blur", "blurry",
        "선명", "선명도", "또렷", "흐림", "블러"
    ],
    ("02. 화질", "02-02. 컬러"): [
        "color", "colour", "vivid", "vibrant", "lifelike", "saturation",
        "컬러", "색감", "생생", "채도"
    ],
    ("02. 화질", "02-03. 밝기"): [
        "bright", "brightness", "dim", "luminous",
        "밝기", "밝음", "어두움"
    ],
    ("02. 화질", "02-04. 명암비"): [
        "contrast", "black level", "black", "white balance",
        "명암", "명암비", "블랙", "검정"
    ],
    ("02. 화질", "02-05. 해상도"): [
        "resolution", "4k", "uhd", "hdr", "detail", "pixel",
        "해상도", "4k", "uhd", "hdr", "디테일", "픽셀"
    ],
    ("02. 화질", "02-06. 움직이는 영상 표현"): [
        "motion", "smooth", "blur", "judder", "stutter", "fast scene",
        "움직임", "모션", "부드러움", "잔상", "버벅"
    ],
    ("02. 화질", "02-08. 시야각"): [
        "angle", "viewing angle", "off angle",
        "시야각", "각도"
    ],
    ("02. 화질", "02-09. 반사율"): [
        "glare", "reflection", "reflective", "anti glare",
        "반사", "반사율", "빛반사", "눈부심"
    ],
    ("02. 화질", "02-10. 화질세팅"): [
        "setting", "mode", "calibration", "preset",
        "설정", "세팅", "모드", "보정"
    ],
    ("02. 화질", "02-10. 화질세팅_(1)화질 모드"): [
        "picture mode", "mode", "preset",
        "화질모드", "모드", "프리셋"
    ],
    ("02. 화질", "02-20. 전반적 화질"): [
        "picture", "image", "visual", "screen quality",
        "화질", "화면", "영상"
    ],
    ("03. 음질", "03-01. 출력"): [
        "volume", "loud", "output", "speaker power",
        "출력", "볼륨", "소리크기"
    ],
    ("03. 음질", "03-02. 선명한 음질"): [
        "clear sound", "clarity", "crisp audio",
        "선명", "맑음", "또렷"
    ],
    ("03. 음질", "03-03. 음질 세팅"): [
        "sound mode", "setting", "equalizer", "eq",
        "설정", "세팅", "모드", "eq"
    ],
    ("03. 음질", "03-04. 서라운드"): [
        "surround", "immersive", "spatial",
        "서라운드", "공간감"
    ],
    ("03. 음질", "03-05. 저음/베이스"): [
        "bass", "low end", "deep sound",
        "저음", "베이스"
    ],
    ("03. 음질", "03-20. 전반적 음질"): [
        "sound", "audio", "speaker",
        "소리", "음질", "사운드"
    ],
    ("04. 디자인", "04-01. 옆면 두께"): [
        "thin", "thickness", "slim",
        "얇음", "두께", "슬림"
    ],
    ("04. 디자인", "04-02. 베젤(프레임) 두께"): [
        "bezel", "frame",
        "베젤", "프레임"
    ],
    ("04. 디자인", "04-03. 스탠드 높이/형태"): [
        "stand", "height", "base",
        "스탠드", "높이", "받침"
    ],
    ("04. 디자인", "04-04. 벽걸이 디자인"): [
        "wall mount", "mount", "flush",
        "벽걸이", "마운트"
    ],
    ("04. 디자인", "04-05. 소재"): [
        "material", "texture",
        "소재", "재질"
    ],
    ("04. 디자인", "04-06. 색상"): [
        "color", "colour",
        "색상", "색"
    ],
    ("04. 디자인", "04-07. 마감"): [
        "finish", "build quality",
        "마감", "완성도"
    ],
    ("04. 디자인", "04-08. 후면부 디자인"): [
        "rear", "back design", "back panel",
        "후면", "뒷면"
    ],
    ("04. 디자인", "04-20. 전반적 디자인"): [
        "beautiful", "stylish", "look", "appearance",
        "예쁨", "외관", "디자인"
    ],
    ("05. 설치/세팅", "05-01. 세팅"): [
        "setup", "set up", "configure",
        "설치", "세팅", "설정"
    ],
    ("05. 설치/세팅", "05-02. 매뉴얼/가이드"): [
        "manual", "guide", "instruction",
        "매뉴얼", "가이드", "설명서"
    ],
    ("05. 설치/세팅", "05-03. 무게"): [
        "weight", "heavy", "light",
        "무게", "무겁", "가볍"
    ],
    ("05. 설치/세팅", "05-04. 선처리"): [
        "cable", "wire", "cable management",
        "선정리", "케이블", "선"
    ],
    ("05. 설치/세팅", "05-05. 각도 조절(벽걸이)"): [
        "angle", "tilt", "swivel",
        "각도", "틸트", "회전"
    ],
    ("05. 설치/세팅", "05-06. 벽걸이 설치용이성"): [
        "wall mount", "mounting",
        "벽걸이", "설치"
    ],
    ("05. 설치/세팅", "05-07. 스탠드 설치용이성"): [
        "stand assembly", "stand install",
        "스탠드", "조립"
    ],
    ("05. 설치/세팅", "05-20. 전반적 설치용이성"): [
        "install", "installation", "easy setup",
        "설치", "세팅", "조립"
    ],
    ("06. 연결성", "06-01. 연결기기 호환성"): [
        "compatible", "compatibility", "device", "devices",
        "호환", "호환성", "기기"
    ],
    ("06. 연결성", "06-02. 무선 연결성"): [
        "wifi", "wireless", "bluetooth", "pairing",
        "와이파이", "무선", "블루투스", "페어링"
    ],
    ("06. 연결성", "06-03. 연결단자 지원/개수"): [
        "hdmi", "usb", "port", "ports",
        "단자", "포트", "hdmi", "usb"
    ],
    ("06. 연결성", "06-04. (편리한)단자 위치"): [
        "port location", "easy access",
        "단자위치", "접근"
    ],
    ("06. 연결성", "06-20. 전반적 연결성"): [
        "connect", "connection",
        "연결", "연결성"
    ],
    ("07. 스마트 사용성", "07-01. 채널/컨텐츠 APP"): [
        "app", "apps", "channel", "content", "streaming", "ott",
        "앱", "채널", "콘텐츠", "스트리밍", "ott"
    ],
    ("07. 스마트 사용성", "07-02. 구동성/구동속도"): [
        "fast", "slow", "lag", "speed", "loading",
        "속도", "빠름", "느림", "로딩", "렉"
    ],
    ("07. 스마트 사용성", "07-02. 구동성/구동속도_(1)TV 전반"): [
        "fast", "slow", "lag", "speed", "tv response",
        "속도", "빠름", "느림", "반응", "렉"
    ],
    ("07. 스마트 사용성", "07-02. 구동성/구동속도_(2)APP/웹전반"): [
        "app speed", "web speed", "lag", "loading",
        "앱속도", "웹속도", "로딩", "렉"
    ],
    ("07. 스마트 사용성", "07-03. 메뉴 구성/UI"): [
        "menu", "ui", "interface", "navigation",
        "메뉴", "ui", "인터페이스", "탐색"
    ],
    ("07. 스마트 사용성", "07-04. SW/OS"): [
        "os", "software", "update", "bug",
        "os", "소프트웨어", "업데이트", "버그"
    ],
    ("07. 스마트 사용성", "07-05. 컨텐츠 탐색 사용성"): [
        "search", "browse", "discover",
        "탐색", "검색", "브라우징"
    ],
    ("07. 스마트 사용성", "07-06. 리모컨 사용성"): [
        "remote", "control", "button", "layout", "pointer", "backlight", "easy",
        "리모컨", "조작", "버튼", "레이아웃", "포인터", "백라이트", "편리"
    ],
    ("07. 스마트 사용성", "07-07. 리모컨 디자인"): [
        "remote design", "remote look",
        "리모컨 디자인", "외관"
    ],
    ("07. 스마트 사용성", "07-08. 음성 인식/조작"): [
        "voice", "speech", "assistant",
        "음성", "음성인식", "음성조작"
    ],
    ("07. 스마트 사용성", "07-09. 게임 기능"): [
        "game", "gaming", "latency",
        "게임", "게이밍", "지연"
    ],
    ("07. 스마트 사용성", "07-10. 부가 기능"): [
        "feature", "features", "extra function",
        "기능", "부가기능"
    ],
    ("07. 스마트 사용성", "07-11. 홈 IoT"): [
        "iot", "smart home",
        "iot", "스마트홈"
    ],
    ("07. 스마트 사용성", "07-12. 모바일 연동"): [
        "mobile", "phone", "cast", "mirror",
        "모바일", "휴대폰", "캐스트", "미러링"
    ],
    ("07. 스마트 사용성", "07-13. 광고"): [
        "ad", "ads", "advertisement",
        "광고"
    ],
    ("07. 스마트 사용성", "07-20. 전반적 스마트 사용성"): [
        "smart", "usability", "easy to use",
        "스마트", "사용성", "편리"
    ],
    ("08. 내구성", "08-01. A/S"): [
        "service", "support", "repair", "warranty",
        "as", "서비스", "지원", "수리", "보증"
    ],
    ("08. 내구성", "08-02. 품질보증기간"): [
        "warranty", "guarantee",
        "보증", "보증기간"
    ],
    ("08. 내구성", "08-03. 잔상"): [
        "ghosting", "burn in", "image retention",
        "잔상", "번인"
    ],
    ("08. 내구성", "08-04. 패널 내구성"): [
        "panel", "durability", "screen life",
        "패널", "내구성"
    ],
    ("08. 내구성", "08-05. 발열"): [
        "heat", "heating", "hot",
        "발열", "뜨거움"
    ],
    ("08. 내구성", "08-20. 전반적 내구성"): [
        "durable", "reliable", "lasting",
        "내구성", "튼튼", "오래감"
    ],
    ("09. 친환경", "09-01. 에너지 효율"): [
        "energy", "efficient", "efficiency", "power saving",
        "에너지", "효율", "절전"
    ],
    ("09. 친환경", "09-02. 친환경 소재"): [
        "eco", "eco-friendly", "material",
        "친환경", "소재"
    ],
    ("10. 가격", "10-01. 가격/가격대비 가치"): [
        "price", "value", "worth", "deal", "budget", "affordable",
        "가격", "가성비", "값어치", "딜", "저렴"
    ],
    ("11. 브랜드", "11-01. 브랜드"): [
        "brand", "samsung", "lg", "sony", "tcl", "hisense",
        "브랜드", "삼성", "엘지", "소니"
    ],
    ("12. 품질 불량", "12-01. 화질 불량"): [
        "defect", "screen issue", "dead pixel", "dark", "blurry",
        "불량", "화질불량", "데드픽셀", "암부", "흐림"
    ],
    ("12. 품질 불량", "12-02. 제품 불량"): [
        "defective", "broken", "fault", "problem",
        "불량", "고장", "문제"
    ],
    ("12. 품질 불량", "12-03. 설치 불량"): [
        "installation issue", "bad install",
        "설치불량", "설치문제"
    ],
    ("12. 품질 불량", "12-04. 기능 불량"): [
        "malfunction", "doesn't work", "failed", "not working",
        "기능불량", "작동안함", "먹통", "고장"
    ],
    ("12. 품질 불량", "12-05. 기타 불량"): [
        "issue", "problem", "fault",
        "불량", "문제", "이슈"
    ],
    ("13. 전반적 평가", "13-01. 전반적 평가"): [
        "overall", "generally", "satisfied", "happy", "purchase", "product",
        "전반적", "전체적", "만족", "구매", "제품"
    ],
}

def get_feature_patterns(cate_1_depth: str, cate_2_depth: str) -> List[str]:
    return COMMON_FEATURE_PATTERNS + CATEGORY_FEATURE_PATTERNS.get((cate_1_depth, cate_2_depth), [])


def has_specific_reason_for_category(text: str, clue_keywords: List[str], cate_1_depth: str, cate_2_depth: str) -> bool:
    memo = clean_text(text).lower()
    if any(clean_text(k).lower() in memo for k in clue_keywords if clean_text(k)):
        return True
    feature_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)
    return any(clean_text(p).lower() in memo for p in feature_patterns if clean_text(p))


def should_be_overall_for_category(text: str, clue_keywords: List[str], generic_terms: List[str], cate_1_depth: str, cate_2_depth: str) -> bool:
    memo = clean_text(text).lower()
    if not memo:
        return False
    if len(memo) > 14:
        return False
    if has_specific_reason_for_category(memo, clue_keywords, cate_1_depth, cate_2_depth):
        return False
    if len(re.findall(r"[A-Za-z가-힣]+", memo)) > 3:
        return False
    return any(clean_text(t).lower() in memo for t in generic_terms if clean_text(t))


In [0]:
# COMMAND ----------
# 4) Load All Eligible Groups And Sample Rows

group_query = f"""
with base as (
    select distinct
        cate_1_depth,
        cate_2_depth,
        sc_measurement,
        memo
    from {SOURCE_TABLE}
    where 1=1
      and cate_1_depth not like '***%'
      and sc_measurement in (1, -1)
      and memo is not null
      and length(trim(memo)) > 0
),
grouped as (
    select
        cate_1_depth,
        cate_2_depth,
        sc_measurement,
        count(*) as group_total_cnt
    from base
    group by 1, 2, 3
)
select *
from grouped
where group_total_cnt >= {MIN_GROUP_SIZE}
order by cate_1_depth, cate_2_depth, sc_measurement
"""

group_df = spark.sql(group_query)

if LIMIT_GROUP_COUNT is not None:
    group_df = group_df.limit(LIMIT_GROUP_COUNT)

group_rows = [r.asDict(recursive=True) for r in group_df.toLocalIterator()]
print(f"[LOAD] total groups to run = {len(group_rows)}")
display(group_df)

In [0]:
# 4.5) Checkpoint / Resume Helpers

PROGRESS_TABLE = f"{SAVE_DB}.category_topic_progress_{PROMPT_VERSION}"
FAILED_GROUP_TABLE = f"{SAVE_DB}.category_topic_failed_group_{PROMPT_VERSION}"

RESUME_FROM_CHECKPOINT = True

def append_table(df, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        print(f"[SAVE] append -> {table_name}")
        df.write.mode("append").format("delta").saveAsTable(table_name)
    else:
        print(f"[SAVE] create -> {table_name}")
        df.write.mode("overwrite").format("delta").saveAsTable(table_name)


def ensure_checkpoint_tables() -> None:
    if not spark.catalog.tableExists(PROGRESS_TABLE):
        spark.createDataFrame(
            [],
            """
            cate_1_depth string,
            cate_2_depth string,
            sc_measurement int,
            stage string,
            status string,
            message string,
            event_ts timestamp
            """
        ).write.mode("overwrite").format("delta").saveAsTable(PROGRESS_TABLE)

    if not spark.catalog.tableExists(FAILED_GROUP_TABLE):
        spark.createDataFrame(
            [],
            """
            cate_1_depth string,
            cate_2_depth string,
            sc_measurement int,
            stage string,
            error_message string,
            event_ts timestamp
            """
        ).write.mode("overwrite").format("delta").saveAsTable(FAILED_GROUP_TABLE)


from pyspark.sql import types as T

PROGRESS_STRUCT = T.StructType([
    T.StructField("cate_1_depth", T.StringType(), True),
    T.StructField("cate_2_depth", T.StringType(), True),
    T.StructField("sc_measurement", T.IntegerType(), True),
    T.StructField("stage", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("message", T.StringType(), True),
    T.StructField("event_ts", T.TimestampType(), True),
])

FAILED_STRUCT = T.StructType([
    T.StructField("cate_1_depth", T.StringType(), True),
    T.StructField("cate_2_depth", T.StringType(), True),
    T.StructField("sc_measurement", T.IntegerType(), True),
    T.StructField("stage", T.StringType(), True),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("event_ts", T.TimestampType(), True),
])

def log_progress(cate_1_depth: str, cate_2_depth: str, sc_measurement: int, stage: str, status: str, message: str = "") -> None:
    row = [(
        clean_text(cate_1_depth),
        clean_text(cate_2_depth),
        int(sc_measurement),
        clean_text(stage),
        clean_text(status),
        clean_text(message),
        pd.Timestamp.now().to_pydatetime(),
    )]
    append_table(spark.createDataFrame(row, schema=PROGRESS_STRUCT), PROGRESS_TABLE)


def log_failure(cate_1_depth: str, cate_2_depth: str, sc_measurement: int, stage: str, error_message: str) -> None:
    row = [(
        clean_text(cate_1_depth),
        clean_text(cate_2_depth),
        int(sc_measurement),
        clean_text(stage),
        clean_text(error_message),
        pd.Timestamp.now().to_pydatetime(),
    )]
    append_table(spark.createDataFrame(row, schema=FAILED_STRUCT), FAILED_GROUP_TABLE)


def get_done_group_keys(stage: str):
    if (not RESUME_FROM_CHECKPOINT) or (not spark.catalog.tableExists(PROGRESS_TABLE)):
        return set()

    rows = (
        spark.table(PROGRESS_TABLE)
        .where(F.col("stage") == F.lit(stage))
        .where(F.col("status") == F.lit("done"))
        .select("cate_1_depth", "cate_2_depth", "sc_measurement")
        .distinct()
        .collect()
    )
    return {(r["cate_1_depth"], r["cate_2_depth"], int(r["sc_measurement"])) for r in rows}


def delete_group_rows(table_name: str, cate_1_depth: str, cate_2_depth: str, sc_measurement: int) -> None:
    if not spark.catalog.tableExists(table_name):
        return

    spark.sql(f"""
        delete from {table_name}
        where cate_1_depth = '{cate_1_depth}'
          and cate_2_depth = '{cate_2_depth}'
          and sc_measurement = {sc_measurement}
    """)


ensure_checkpoint_tables()

print(f"[CHECKPOINT] progress_table={PROGRESS_TABLE}")
print(f"[CHECKPOINT] failed_group_table={FAILED_GROUP_TABLE}")

def load_group_sample_df(cate_1_depth: str, cate_2_depth: str, sc_measurement: int, max_rows: int):
    query = f"""
    with base as (
        select distinct memo
        from {SOURCE_TABLE}
        where 1=1
          and cate_1_depth = '{cate_1_depth}'
          and cate_2_depth = '{cate_2_depth}'
          and sc_measurement = {sc_measurement}
          and memo is not null
          and length(trim(memo)) > 0
    ),
    sampled as (
        select
            memo,
            row_number() over (
                order by md5(
                    concat(
                        coalesce(memo, ''),
                        '||',
                        '{cate_1_depth}',
                        '||',
                        '{cate_2_depth}',
                        '||',
                        cast({sc_measurement} as string),
                        '||seed_20260420'
                    )
                )
            ) as rn
        from base
    )
    select memo
    from sampled
    where rn <= {max_rows}
    order by rn
    """
    return spark.sql(query).withColumn("_row_id", F.monotonically_increasing_id()).cache()

In [0]:
# 5) Build Rule Profile Table For All Groups (checkpoint / resume)
spark.sql(f"truncate table {FAILED_GROUP_TABLE}")
print(f"[RESET] truncated {FAILED_GROUP_TABLE}")

def build_rule_profile_messages(cate_1_depth: str, cate_2_depth: str, sc_measurement: int, sample_memos: List[str]) -> List[Dict[str, str]]:
    label = sc_label(sc_measurement)
    overall = overall_label(sc_measurement)
    category_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)

    system = f"""
You are a VOC rule designer for TV review topic classification.

Category:
- cate_1_depth = {cate_1_depth}
- cate_2_depth = {cate_2_depth}
- polarity = {label}

Design outputs:
- overall_topic_label
- clue_keywords
- generic_terms
- overall_usage_rule
- specific_reason_rule
- non_overall_examples

Rules:
- clue_keywords must be specialized for this category.
- generic_terms must be suitable for this polarity only.
- {overall} should be extremely narrow.
- {overall} must be limited to ultra-short sentiment-only memos where no usable reason can be inferred.
- If a memo contains any usable reason, feature, benefit, complaint, or context clue, it must not be treated as {overall}.
- category fallback feature patterns:
  {json.dumps(category_patterns[:80], ensure_ascii=False)}

Return only JSON:
{{
  "overall_topic_label": "",
  "clue_keywords": [""],
  "generic_terms": [""],
  "overall_usage_rule": "",
  "specific_reason_rule": "",
  "non_overall_examples": [""]
}}
"""
    user = "Review memos:\n" + "\n".join([f"- {clean_text(m)}" for m in sample_memos])
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


done_rule_groups = get_done_group_keys("rule_profile")
total_groups = len(group_rows)

print(f"[RULE PROFILE] total_groups={total_groups}, done={len(done_rule_groups)}, pending={total_groups - len(done_rule_groups)}")

for idx, g in enumerate(group_rows, start=1):
    key = (g["cate_1_depth"], g["cate_2_depth"], int(g["sc_measurement"]))

    if key in done_rule_groups:
        print(f"[RULE PROFILE SKIP] {idx}/{total_groups} already done - {key}")
        continue

    print(f"[RULE PROFILE START] {idx}/{total_groups} - {key}")
    group_start_ts = time.time()

    try:
        rule_sample_df = load_group_sample_df(
            g["cate_1_depth"],
            g["cate_2_depth"],
            g["sc_measurement"],
            MAX_RULE_SAMPLE_ROWS
        )
        rule_sample_memos = [r["memo"] for r in rule_sample_df.select("memo").toLocalIterator()]

        rule_result = call_llm(
            build_rule_profile_messages(
                g["cate_1_depth"],
                g["cate_2_depth"],
                g["sc_measurement"],
                rule_sample_memos,
            )
        )

        rule_pdf = pd.DataFrame(
            [{
                "cate_1_depth": g["cate_1_depth"],
                "cate_2_depth": g["cate_2_depth"],
                "sc_measurement": g["sc_measurement"],
                "group_total_cnt": g["group_total_cnt"],
                "overall_topic_label": clean_text(rule_result.get("overall_topic_label")) or overall_label(g["sc_measurement"]),
                "clue_keywords_json": json.dumps(rule_result.get("clue_keywords", [])[:80], ensure_ascii=False),
                "generic_terms_json": json.dumps(rule_result.get("generic_terms", [])[:30], ensure_ascii=False),
                "overall_usage_rule": clean_text(rule_result.get("overall_usage_rule")),
                "specific_reason_rule": clean_text(rule_result.get("specific_reason_rule")),
                "non_overall_examples_json": json.dumps(rule_result.get("non_overall_examples", [])[:12], ensure_ascii=False),
            }]
        )

        delete_group_rows(RULE_PROFILE_TABLE, g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"])
        append_table(spark.createDataFrame(rule_pdf), RULE_PROFILE_TABLE)
        log_progress(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], "rule_profile", "done", "saved rule profile")

        rule_sample_df.unpersist()

        elapsed_sec = round(time.time() - group_start_ts, 2)
        progress_pct = round(idx / total_groups * 100, 2)
        print(f"[RULE PROFILE DONE] {idx}/{total_groups} ({progress_pct}%) elapsed_sec={elapsed_sec}")

        time.sleep(RATE_LIMIT_SECONDS)

    except Exception as exc:
        log_failure(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], "rule_profile", repr(exc))
        print(f"[RULE PROFILE FAILED] {idx}/{total_groups} error={repr(exc)}")

rule_profile_df = spark.table(RULE_PROFILE_TABLE)
display(rule_profile_df)

In [0]:
# COMMAND ----------
# 6) Build Topic Pool Table For All Groups (checkpoint / resume)

rule_map = {
    (r["cate_1_depth"], r["cate_2_depth"], int(r["sc_measurement"])): r.asDict(recursive=True)
    for r in spark.table(RULE_PROFILE_TABLE).toLocalIterator()
}

def build_topic_pool_messages(cate_1_depth: str, cate_2_depth: str, sc_measurement: int, sample_memos: List[str], rule_row: Dict[str, Any]) -> List[Dict[str, str]]:
    overall_topic_label = clean_text(rule_row["overall_topic_label"])
    clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []
    non_overall_examples = json.loads(rule_row["non_overall_examples_json"]) if rule_row["non_overall_examples_json"] else []
    category_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)

    system = f"""
You are a VOC taxonomy designer for TV review topic classification.

Category:
- cate_1_depth = {cate_1_depth}
- cate_2_depth = {cate_2_depth}
- polarity = {sc_label(sc_measurement)}

Rules:
- Final topic count must be at least {MIN_FINAL_TOPICS} and at most {MAX_FINAL_TOPICS}.
- One mandatory topic must be '{overall_topic_label}'.
- {clean_text(rule_row["overall_usage_rule"])}
- {clean_text(rule_row["specific_reason_rule"])}
- clue keywords:
  {json.dumps(clue_keywords, ensure_ascii=False)}
- category fallback feature patterns:
  {json.dumps(category_patterns[:80], ensure_ascii=False)}
- non-overall examples:
  {json.dumps(non_overall_examples, ensure_ascii=False)}
- Topic labels must be Korean.
- Topic should be concise noun-like or service/function-like wording.
- Description should be short Korean.
- Avoid duplicates and near-synonyms.

Return only JSON:
{{
  "topics": [
    {{
      "topic": "",
      "description": "",
      "representative_memos": [""]
    }}
  ]
}}
"""
    user = "Sample memos:\n" + "\n".join([f"- {clean_text(m)}" for m in sample_memos])
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


done_topic_groups = get_done_group_keys("topic_pool")
total_groups = len(group_rows)

print(f"[TOPIC POOL] total_groups={total_groups}, done={len(done_topic_groups)}, pending={total_groups - len(done_topic_groups)}")

for idx, g in enumerate(group_rows, start=1):
    key = (g["cate_1_depth"], g["cate_2_depth"], int(g["sc_measurement"]))

    if key in done_topic_groups:
        print(f"[TOPIC POOL SKIP] {idx}/{total_groups} already done - {key}")
        continue

    if key not in rule_map:
        print(f"[TOPIC POOL SKIP] missing rule profile - {key}")
        continue

    print(f"[TOPIC POOL START] {idx}/{total_groups} - {key}")
    group_start_ts = time.time()

    try:
        sample_df = load_group_sample_df(
            g["cate_1_depth"],
            g["cate_2_depth"],
            g["sc_measurement"],
            MAX_SAMPLE_ROWS
        )
        sample_memos = [r["memo"] for r in sample_df.select("memo").toLocalIterator()]

        topic_pool_result = call_llm(
            build_topic_pool_messages(
                g["cate_1_depth"],
                g["cate_2_depth"],
                g["sc_measurement"],
                sample_memos,
                rule_map[key],
            )
        )

        topic_rows = []
        for order_no, topic in enumerate(topic_pool_result.get("topics", [])[:MAX_FINAL_TOPICS], start=1):
            if not isinstance(topic, dict):
                continue
            topic_rows.append(
                {
                    "cate_1_depth": g["cate_1_depth"],
                    "cate_2_depth": g["cate_2_depth"],
                    "sc_measurement": g["sc_measurement"],
                    "topic_order": order_no,
                    "topic": clean_text(topic.get("topic")),
                    "description": clean_text(topic.get("description")),
                    "representative_memos_json": json.dumps(topic.get("representative_memos", [])[:5], ensure_ascii=False),
                }
            )

        delete_group_rows(TOPIC_POOL_TABLE, g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"])
        append_table(spark.createDataFrame(pd.DataFrame(topic_rows)), TOPIC_POOL_TABLE)
        log_progress(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], "topic_pool", "done", "saved topic pool")

        sample_df.unpersist()

        elapsed_sec = round(time.time() - group_start_ts, 2)
        progress_pct = round(idx / total_groups * 100, 2)
        print(f"[TOPIC POOL DONE] {idx}/{total_groups} ({progress_pct}%) elapsed_sec={elapsed_sec}")

        time.sleep(RATE_LIMIT_SECONDS)

    except Exception as exc:
        log_failure(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], "topic_pool", repr(exc))
        print(f"[TOPIC POOL FAILED] {idx}/{total_groups} error={repr(exc)}")

topic_pool_df = spark.table(TOPIC_POOL_TABLE)
display(topic_pool_df)


In [0]:
# COMMAND ----------
# 7) Classify Sample Rows For All Groups (checkpoint / resume)

rule_map = {
    (r["cate_1_depth"], r["cate_2_depth"], int(r["sc_measurement"])): r.asDict(recursive=True)
    for r in spark.table(RULE_PROFILE_TABLE).toLocalIterator()
}

topic_pool_group_map = {}
for row in spark.table(TOPIC_POOL_TABLE).toLocalIterator():
    key = (row["cate_1_depth"], row["cate_2_depth"], int(row["sc_measurement"]))
    topic_pool_group_map.setdefault(key, []).append(
        {
            "topic": row["topic"],
            "description": row["description"],
        }
    )

def build_classify_messages(batch_rows: List[Dict[str, Any]], topic_pool_payload: List[Dict[str, Any]], rule_row: Dict[str, Any], cate_1_depth: str, cate_2_depth: str, sc_measurement: int) -> List[Dict[str, str]]:
    clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []
    non_overall_examples = json.loads(rule_row["non_overall_examples_json"]) if rule_row["non_overall_examples_json"] else []
    category_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)

    system = f"""
You are a VOC classifier for TV review topic classification.

Classify each memo into exactly one topic from the fixed topic list.
Every memo must belong to exactly one topic.

Rules:
- The task is to identify WHY the writer evaluated {cate_2_depth} as {sc_label(sc_measurement)}.
- Overall topic is '{clean_text(rule_row["overall_topic_label"])}'.
- {clean_text(rule_row["overall_usage_rule"])}
- {clean_text(rule_row["specific_reason_rule"])}
- clue keywords:
  {json.dumps(clue_keywords, ensure_ascii=False)}
- category fallback feature patterns:
  {json.dumps(category_patterns[:80], ensure_ascii=False)}
- non-overall examples:
  {json.dumps(non_overall_examples, ensure_ascii=False)}
- Do not invent any new topic.
- explanation must be a short Korean sentence.

Return only JSON:
{{
  "results": [
    {{
      "row_id": "",
      "topic": "",
      "explanation": ""
    }}
  ]
}}
"""
    user = (
        "Fixed topics:\n"
        + json.dumps(topic_pool_payload, ensure_ascii=False)
        + "\n\nMemos:\n"
        + json.dumps(
            [{"row_id": str(r["_row_id"]), "memo": clean_text(r["memo"])} for r in batch_rows],
            ensure_ascii=False,
        )
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


def build_reclassify_without_overall_message(row: Dict[str, Any], topic_pool_payload: List[Dict[str, Any]], rule_row: Dict[str, Any], cate_1_depth: str, cate_2_depth: str) -> List[Dict[str, str]]:
    overall_topic = clean_text(rule_row["overall_topic_label"])
    specific_topic_payload = [topic for topic in topic_pool_payload if clean_text(topic["topic"]) != overall_topic]
    clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []
    category_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)

    system = f"""
You are a VOC classifier for TV review topic classification.

This memo was incorrectly over-generalized as '{overall_topic}'.
Reclassify it using only the non-general topics below.

Rules:
- Choose exactly one non-general topic.
- The memo contains a specific reason and must not remain as '{overall_topic}'.
- clue keywords:
  {json.dumps(clue_keywords, ensure_ascii=False)}
- category fallback feature patterns:
  {json.dumps(category_patterns[:80], ensure_ascii=False)}
- explanation must be a short Korean sentence.

Return only JSON:
{{
  "topic": "",
  "explanation": ""
}}
"""
    user = (
        "Allowed non-general topics:\n"
        + json.dumps(specific_topic_payload, ensure_ascii=False)
        + "\n\nMemo:\n"
        + json.dumps({"memo": clean_text(row["memo"])}, ensure_ascii=False)
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


done_classify_groups = get_done_group_keys("classify_sample")
total_groups = len(group_rows)

print(f"[CLASSIFY] total_groups={total_groups}, done={len(done_classify_groups)}, pending={total_groups - len(done_classify_groups)}")

for idx, g in enumerate(group_rows, start=1):
    key = (g["cate_1_depth"], g["cate_2_depth"], int(g["sc_measurement"]))

    if key in done_classify_groups:
        print(f"[CLASSIFY SKIP] {idx}/{total_groups} already done - {key}")
        continue

    if key not in rule_map or key not in topic_pool_group_map:
        print(f"[CLASSIFY SKIP] missing stage1 data - {key}")
        continue

    print(f"[CLASSIFY START] {idx}/{total_groups} - {key}")
    group_start_ts = time.time()

    try:
        rule_row = rule_map[key]
        topic_payload = topic_pool_group_map[key]
        clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []
        generic_terms = json.loads(rule_row["generic_terms_json"]) if rule_row["generic_terms_json"] else []

        sample_df = load_group_sample_df(
            g["cate_1_depth"],
            g["cate_2_depth"],
            g["sc_measurement"],
            MAX_SAMPLE_ROWS
        )
        source_rows = [r.asDict(recursive=True) for r in sample_df.toLocalIterator()]
        total_batches = (len(source_rows) + CLASSIFY_BATCH_SIZE - 1) // CLASSIFY_BATCH_SIZE

        classified_rows = []

        for batch_no, batch in enumerate(chunk_list(source_rows, CLASSIFY_BATCH_SIZE), start=1):
            batch_start_ts = time.time()
            print(f"[CLASSIFY BATCH START] group={idx}/{total_groups}, batch={batch_no}/{total_batches}, rows={len(batch)}")

            batch_result = call_llm(
                build_classify_messages(
                    batch,
                    topic_payload,
                    rule_row,
                    g["cate_1_depth"],
                    g["cate_2_depth"],
                    g["sc_measurement"],
                )
            )

            result_map = {
                str(item.get("row_id")): item
                for item in batch_result.get("results", [])
                if isinstance(item, dict)
            }

            for row in batch:
                mapped = result_map.get(str(row["_row_id"]), {})
                topic_raw = clean_text(mapped.get("topic"))
                explanation_raw = clean_text(mapped.get("explanation"))

                if (
                    topic_raw == clean_text(rule_row["overall_topic_label"])
                    and has_specific_reason_for_category(row["memo"], clue_keywords, g["cate_1_depth"], g["cate_2_depth"])
                    and not should_be_overall_for_category(row["memo"], clue_keywords, generic_terms, g["cate_1_depth"], g["cate_2_depth"])
                ):
                    retry_result = call_llm(
                        build_reclassify_without_overall_message(
                            row,
                            topic_payload,
                            rule_row,
                            g["cate_1_depth"],
                            g["cate_2_depth"],
                        )
                    )
                    retry_topic = clean_text(retry_result.get("topic"))
                    retry_explanation = clean_text(retry_result.get("explanation"))
                    if retry_topic:
                        topic_raw = retry_topic
                    if retry_explanation:
                        explanation_raw = retry_explanation

                classified_rows.append(
                    {
                        "_row_id": row["_row_id"],
                        "cate_1_depth": g["cate_1_depth"],
                        "cate_2_depth": g["cate_2_depth"],
                        "sc_measurement": g["sc_measurement"],
                        "memo": row["memo"],
                        "topic_raw": topic_raw,
                        "explanation_raw": explanation_raw,
                    }
                )

            batch_elapsed_sec = round(time.time() - batch_start_ts, 2)
            print(f"[CLASSIFY BATCH DONE] group={idx}/{total_groups}, batch={batch_no}/{total_batches}, elapsed_sec={batch_elapsed_sec}")

            time.sleep(RATE_LIMIT_SECONDS)

        classified_group_df = spark.createDataFrame(pd.DataFrame(classified_rows))

        topic_stats_df = classified_group_df.groupBy("topic_raw").agg(F.count("*").alias("topic_cnt"))
        rare_topics = [r["topic_raw"] for r in topic_stats_df.where(F.col("topic_cnt") < F.lit(MIN_ROWS_PER_TOPIC)).collect()]
        rare_topics_sorted = sorted([t for t in rare_topics if clean_text(t)])
        rare_topic_label = f"기타({', '.join(rare_topics_sorted[:3])})" if rare_topics_sorted else "기타"

        topic_desc_map = {
            topic["topic"]: topic["description"]
            for topic in topic_payload
        }

        final_rows = []
        for row in classified_group_df.toLocalIterator():
            row_dict = row.asDict(recursive=True)
            raw_topic = clean_text(row_dict["topic_raw"])
            raw_expl = clean_text(row_dict["explanation_raw"])

            if raw_topic in rare_topics_sorted:
                final_topic = rare_topic_label
                final_explanation = f"희소 주제 묶음: {raw_topic}" if raw_topic else "희소 주제 묶음"
            else:
                final_topic = raw_topic if raw_topic else "기타"
                final_explanation = raw_expl or topic_desc_map.get(final_topic, "")

            final_rows.append(
                {
                    "_row_id": row_dict["_row_id"],
                    "cate_1_depth": row_dict["cate_1_depth"],
                    "cate_2_depth": row_dict["cate_2_depth"],
                    "sc_measurement": row_dict["sc_measurement"],
                    "memo": row_dict["memo"],
                    "topic": final_topic,
                    "description": final_explanation,
                }
            )

        delete_group_rows(DETAIL_TABLE, g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"])
        append_table(spark.createDataFrame(pd.DataFrame(final_rows)), DETAIL_TABLE)
        log_progress(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], "classify_sample", "done", "saved sample detail")

 sandbox.t_ai_proficiency_new       sample_df.unpersist()

        elapsed_sec = round(time.time() - group_start_ts, 2)
        progress_pct = round(idx / total_groups * 100, 2)
        print(f"[CLASSIFY DONE] {idx}/{total_groups} ({progress_pct}%) elapsed_sec={elapsed_sec}")

    except Exception as exc:
        log_failure(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], "classify_sample", repr(exc))
        print(f"[CLASSIFY FAILED] {idx}/{total_groups} error={repr(exc)}")

detail_df = spark.table(DETAIL_TABLE)
display(detail_df.limit(20))


In [0]:
# COMMAND ----------
# 1) 현재 detail 테이블 기본 현황 확인

print(f"[DETAIL_TABLE] {DETAIL_TABLE}")

detail_exists = spark.catalog.tableExists(DETAIL_TABLE)
print(f"[TABLE EXISTS] {detail_exists}")

if detail_exists:
    detail_df_current = spark.table(DETAIL_TABLE)
    print(f"[DETAIL ROW COUNT] {detail_df_current.count()}")
    display(detail_df_current.limit(20))

# COMMAND ----------
# 2) 현재 저장된 그룹별 row 수 확인

if detail_exists:
    progress_by_group_df = (
        detail_df_current.groupBy("cate_1_depth", "cate_2_depth", "sc_measurement")
        .agg(F.count("*").alias("saved_row_count"))
        .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement")
    )

    print(f"[SAVED GROUP COUNT] {progress_by_group_df.count()}")
    display(progress_by_group_df)
# COMMAND ----------
# 3) 전체 대상 그룹 대비 완료/미완료 그룹 확인

group_query = f"""
with base as (
    select distinct
        cate_1_depth,
        cate_2_depth,
        sc_measurement,
        memo
    from {SOURCE_TABLE}
    where 1=1
      and cate_1_depth not like '***%'
      and sc_measurement in (1, -1)
      and memo is not null
      and length(trim(memo)) > 0
),
grouped as (
    select
        cate_1_depth,
        cate_2_depth,
        sc_measurement,
        count(*) as group_total_cnt
    from base
    group by 1, 2, 3
)
select *
from grouped
where group_total_cnt >= {MIN_GROUP_SIZE}
order by cate_1_depth, cate_2_depth, sc_measurement
"""

all_group_df = spark.sql(group_query)

if detail_exists:
    done_group_df = (
        detail_df_current
        .select("cate_1_depth", "cate_2_depth", "sc_measurement")
        .distinct()
        .withColumn("is_done", F.lit(1))
    )

    group_status_df = (
        all_group_df.join(
            done_group_df,
            on=["cate_1_depth", "cate_2_depth", "sc_measurement"],
            how="left"
        )
        .withColumn("is_done", F.coalesce(F.col("is_done"), F.lit(0)))
        .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement")
    )

    print(f"[ALL GROUPS] {all_group_df.count()}")
    print(f"[DONE GROUPS] {group_status_df.where(F.col('is_done') == 1).count()}")
    print(f"[PENDING GROUPS] {group_status_df.where(F.col('is_done') == 0).count()}")

    display(group_status_df)

# COMMAND ----------
# 4) 미완료 그룹만 보기

if detail_exists:
    pending_group_df = (
        group_status_df
        .where(F.col("is_done") == 0)
        .select("cate_1_depth", "cate_2_depth", "sc_measurement", "group_total_cnt")
        .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement")
    )

    display(pending_group_df)


# COMMAND ----------
# 5) 현재까지 저장된 topic 분포 확인

if detail_exists:
    topic_dist_df = (
        detail_df_current.groupBy("topic")
        .agg(F.count("*").alias("review_count"))
        .withColumn("total_count", F.sum("review_count").over(Window.partitionBy()))
        .withColumn("review_share", F.round(F.col("review_count") / F.col("total_count"), 4))
        .withColumn("review_share_pct", F.round(F.col("review_share") * 100, 2))
        .orderBy(F.desc("review_count"), "topic")
    )

    display(topic_dist_df)

# COMMAND ----------
# 6) 전반적 긍정 / 전반적 부정 비중 확인

if detail_exists:
    overall_df = (
        detail_df_current
        .where(F.col("topic").isin("전반적 긍정", "전반적 부정"))
        .groupBy("cate_1_depth", "cate_2_depth", "sc_measurement", "topic")
        .agg(F.count("*").alias("review_count"))
        .orderBy(F.desc("review_count"))
    )

    display(overall_df)

In [0]:
save_table(group_status_df, f"{SAVE_DB}.category_topic_group_status")

In [0]:
# COMMAND ----------
# 8) Rebuild Summary Table

detail_df = spark.table(DETAIL_TABLE)

summary_df = (
    detail_df.groupBy("cate_1_depth", "cate_2_depth", "sc_measurement", "topic")
    .agg(F.count("*").alias("review_count"))
    .withColumn(
        "group_total_count",
        F.sum("review_count").over(
            Window.partitionBy("cate_1_depth", "cate_2_depth", "sc_measurement")
        ),
    )
    .withColumn("review_share", F.round(F.col("review_count") / F.col("group_total_count"), 4))
    .withColumn("review_share_pct", F.round(F.col("review_share") * 100, 2))
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement", F.desc("review_count"))
)

save_table(summary_df, SUMMARY_TABLE)

print("[SUMMARY DONE]")
display(summary_df)

print(f"""
[RESUME GUIDE]
- rule profile부터 재개:
  5번 셀부터 다시 실행
- topic pool부터 재개:
  6번 셀부터 다시 실행
- classify부터 재개:
  7번 셀부터 다시 실행
- summary만 다시:
  8번 셀만 실행

checkpoint tables:
- {PROGRESS_TABLE}
- {FAILED_GROUP_TABLE}
""")


## 개요

`all_groups` 코드는 Buzzmetrics VOC에서  
모든 `(cate_1_depth, cate_2_depth, sc_measurement)` 그룹에 대해  
샘플 기반 주제 설계와 샘플 분류 결과를 생성하고 저장하는 파이프라인입니다.

대상은 다음 조건을 만족하는 그룹입니다.

- `cate_1_depth not like '***%'`
- `sc_measurement in (1, -1)`
- `memo is not null`
- 그룹별 리뷰 수 `>= 100`

즉, 중립(`0`)은 제외하고  
긍정/부정 그룹만 대상으로 합니다.

---

## 목적

이 코드는 각 그룹별로:

- `전반적 긍정 / 전반적 부정`을 매우 좁게 정의하고
- 구체적 이유를 설명하는 topic pool을 만들고
- 샘플 메모를 그 topic으로 분류한 뒤
- 주제 분포를 테이블로 저장

하는 데 목적이 있습니다.

즉, 아직 전체 모집단 전수 분류보다는  
**그룹별 샘플 1000건 기준의 taxonomy 설계 + 1차 분류 결과 생성**에 가깝습니다.

---

## 처리 단계

### 1. 그룹 목록 추출
원천 테이블에서 조건에 맞는 모든 그룹을 찾습니다.

그룹 기준:
- `cate_1_depth`
- `cate_2_depth`
- `sc_measurement`

---

### 2. 그룹별 샘플 메모 추출
각 그룹마다 최대 `1000건`의 메모를 고정 seed 기반으로 샘플링합니다.

특징:
- 같은 데이터와 같은 seed면 재현 가능한 샘플
- `rule profile` 생성용으로는 최대 `800건` 사용 가능

---

### 3. Rule Profile 생성
각 그룹의 샘플 메모를 보고 LLM이 다음 값을 생성합니다.

- `overall_topic_label`
- `clue_keywords`
- `generic_terms`
- `overall_usage_rule`
- `specific_reason_rule`
- `non_overall_examples`

여기서 핵심은:
- `전반적 긍정 / 전반적 부정`을 매우 좁게 유지하고
- 구체적인 이유가 있는 메모는 overall로 보내지 않도록 규칙을 만드는 것입니다.

---

### 4. Topic Pool 생성
같은 그룹의 샘플 메모와 rule profile을 바탕으로  
LLM이 고정 topic pool을 생성합니다.

규칙:
- 최소 `7개`, 최대 `17개`
- `전반적 긍정` 또는 `전반적 부정` 포함
- topic은 한국어 기준
- 중복/유사 주제 최소화

---

### 5. 샘플 메모 분류
샘플 1000건을 생성된 topic pool에 분류합니다.

분류 시에도:
- `clue_keywords`
- `generic_terms`
- `overall_usage_rule`
- `specific_reason_rule`
- 카테고리별 fallback feature pattern

을 사용합니다.

즉, overall topic이 너무 넓게 잡히지 않도록  
후처리 guardrail이 같이 작동합니다.

---

### 6. 희소 주제 병합
한 그룹 안에서 특정 topic의 리뷰 수가 `10건 미만`이면  
그 topic은 독립 주제로 유지하지 않고 `기타(...)`로 병합합니다.

예:
- `기타(주제A, 주제B, 주제C)`

이렇게 해서 topic이 지나치게 잘게 쪼개지지 않도록 관리합니다.

---

### 7. 결과 저장
최종적으로 아래 테이블을 저장합니다.

- `RULE_PROFILE_TABLE`
- `TOPIC_POOL_TABLE`
- `DETAIL_TABLE`
- `SUMMARY_TABLE`

---

## Checkpoint / Resume 구조

이 버전은 오래 걸리는 전체 그룹 실행을 위해  
그룹별 checkpoint/resume 구조를 사용합니다.

추가 테이블:
- `PROGRESS_TABLE`
- `FAILED_GROUP_TABLE`

작동 방식:
- 각 그룹의 stage 완료 시 `PROGRESS_TABLE`에 `done` 기록
- 실행 중 실패한 그룹은 `FAILED_GROUP_TABLE`에 기록
- 재실행 시 이미 `done`인 그룹은 자동 skip
- 아직 완료되지 않은 그룹만 계속 수행

즉:
- 중간에 클러스터가 끊겨도
- 다음 실행에서 처음부터 전부 다시 돌지 않고
- 남은 그룹만 이어서 처리할 수 있습니다.

---

## stage별 재시작 방식

설정값으로 특정 stage부터 다시 시작할 수 있습니다.

예:
- `rule profile`부터 다시
- `topic pool`부터 다시
- `classify`부터 다시
- `summary`만 다시 생성

즉, 앞 단계 결과 테이블이 이미 있으면  
뒤 단계만 이어서 실행할 수 있습니다.

---

## 진행 상황 확인

실행 중에는 다음 정보가 출력됩니다.

- 현재 몇 번째 그룹인지
- 어떤 `(cate_1_depth, cate_2_depth, sc_measurement)` 그룹인지
- 해당 그룹 처리 완료 여부
- 그룹별 소요 시간
- 분류 단계에서는 batch 진행 로그

그래서 전체 그룹 중 어디까지 진행됐는지  
중간중간 확인할 수 있습니다.

---

## 한 줄 요약

`all_groups` 코드는  
**모든 VOC 카테고리 그룹에 대해 샘플 기반 규칙 생성, topic pool 생성, 샘플 분류, 희소 주제 병합, 결과 저장을 수행하고, 그룹별 checkpoint/resume로 중단 후 이어서 실행할 수 있게 만든 전체 배치용 구조**입니다.